# 06 — Threshold Optimization

This notebook tunes the anomaly threshold for the Isolation Forest model to maximize
F1-score on the test set. It sweeps a range of thresholds, plots precision-recall curves,
and saves the optimal threshold for use in the detection pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, recall_score, f1_score, precision_recall_curve, auc
)
import joblib
import json
import os
import sys

In [ ]:
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))

## 1. Load Models and Data

In [ ]:
iso_model = joblib.load("../models/isolation_forest.pkl")
rf_model = joblib.load("../models/random_forest.pkl")
scaler = joblib.load("../models/scaler.pkl")
label_encoder = joblib.load("../models/label_encoder.pkl")

test_data = pd.read_csv("../data/processed/test.csv")
feature_cols = [c for c in test_data.columns if c not in ["Label", "label"]]
X_test = test_data[feature_cols].values
y_test = test_data["Label"].values

print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_test.shape[1]}")
print(f"Unique labels: {np.unique(y_test)}")

## 2. Generate Anomaly Scores

In [ ]:
scores = iso_model.decision_function(X_test)

# Convert to anomaly scores (higher = more anomalous)
anomaly_scores = -scores

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(anomaly_scores, bins=100, edgecolor="black", alpha=0.7)
axes[0].set_title("Anomaly Score Distribution (All Samples)")
axes[0].set_xlabel("Anomaly Score")
axes[0].set_ylabel("Count")

# By class: BENIGN vs Attack
is_attack = y_test != "BENIGN"
axes[1].hist(anomaly_scores[~is_attack], bins=80, alpha=0.6, label="BENIGN", density=True)
axes[1].hist(anomaly_scores[is_attack], bins=80, alpha=0.6, label="Attack", density=True)
axes[1].set_title("Anomaly Score by Class")
axes[1].set_xlabel("Anomaly Score")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Sweep Thresholds

In [ ]:
# Binary ground truth: 1 = attack, 0 = benign
y_binary = (y_test != "BENIGN").astype(int)

thresholds = np.arange(0.1, 0.95, 0.05)
results = []

for t in thresholds:
    preds = (anomaly_scores >= t).astype(int)
    p = precision_score(y_binary, preds, zero_division=0)
    r = recall_score(y_binary, preds, zero_division=0)
    f1 = f1_score(y_binary, preds, zero_division=0)
    results.append({"threshold": t, "precision": p, "recall": r, "f1": f1})

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.plot(df_results["threshold"], df_results["precision"], "o-", label="Precision")
plt.plot(df_results["threshold"], df_results["recall"], "s-", label="Recall")
plt.plot(df_results["threshold"], df_results["f1"], "^-", label="F1")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Metrics vs Anomaly Threshold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Precision-Recall Curve

In [ ]:
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_binary, anomaly_scores)
pr_auc = auc(recall_vals, precision_vals)

# Find optimal threshold (max F1 on PR curve)
f1_scores_pr = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-10)
optimal_idx = np.argmax(f1_scores_pr)
best_threshold_pr = pr_thresholds[optimal_idx] if optimal_idx < len(pr_thresholds) else pr_thresholds[-1]

plt.figure(figsize=(10, 7))
plt.plot(recall_vals, precision_vals, "b-", linewidth=2, label=f"PR curve (AUC = {pr_auc:.4f})")
plt.scatter(recall_vals[optimal_idx], precision_vals[optimal_idx], c="red", s=120, zorder=5,
            label=f"Optimal threshold = {best_threshold_pr:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend(loc="best")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Optimal Threshold

In [ ]:
# Use sweep results for best F1
best_row = df_results.loc[df_results["f1"].idxmax()]
best_threshold = best_row["threshold"]
best_f1 = best_row["f1"]
best_precision = best_row["precision"]
best_recall = best_row["recall"]

print("=" * 50)
print("OPTIMAL THRESHOLD RESULTS")
print("=" * 50)
print(f"  Best Threshold:  {best_threshold:.4f}")
print(f"  Best F1-Score:   {best_f1:.4f}")
print(f"  Precision:       {best_precision:.4f}")
print(f"  Recall:          {best_recall:.4f}")
print(f"  PR-AUC:          {pr_auc:.4f}")
print("=" * 50)

## 6. Save Thresholds

In [ ]:
os.makedirs("../models/metadata", exist_ok=True)

thresholds_config = {
    "anomaly_threshold": float(best_threshold),
    "f1_score": float(best_f1),
    "precision": float(best_precision),
    "recall": float(best_recall),
    "pr_auc": float(pr_auc),
}

with open("../models/metadata/thresholds.json", "w") as f:
    json.dump(thresholds_config, f, indent=2)

print("Thresholds saved to ../models/metadata/thresholds.json")
print(json.dumps(thresholds_config, indent=2))